# 00/ Synthetic Dataset Generation

This notebook generates a synthetic supermarket transaction dataset 
based on Ecuadorian consumer behavior (Supermaxi / Mi Comisariato context).

**Parameters:**
- 10,000 transactions
- 6 product categories
- 2 to 4 categories purchased per transaction
- 3 cities: Quito, Guayaquil, Cuenca
- Purchase patterns calibrated by time slot and day of week

Data is stored in a SQLite database at `data/retail_ecuador.db`.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

np.random.seed(42)

# General parameters
N = 10000

categories = [
    "Dairy products and eggs",
    "Meats and sausages",
    "Fruits and vegetables",
    "Snacks and drinks",
    "Home cleaning",
    "Personal care"
]

cities = ["Quito", "Guayaquil", "Cuenca"]
customer_type = ["Regular", "New"]
payment_method = ["Cash", "Credit card", "Debit card"]

# Probability of buying 2, 3, or 4 categories per transaction
n_categories_prob = [0.45, 0.35, 0.20]

# Category weights by time slot (Ecuadorian pattern)
weights_by_slot = {
    "Morning":   [0.30, 0.20, 0.25, 0.08, 0.10, 0.07],
    "Afternoon": [0.15, 0.20, 0.20, 0.20, 0.15, 0.10],
    "Evening":   [0.10, 0.15, 0.10, 0.30, 0.20, 0.15]
}

print("Parameters defined successfully")
print(f"Categories: {categories}")

## Dataset generation

Each transaction contains 2 to 4 product categories, simulating a real
supermarket visit where customers buy multiple items. Category combinations
are weighted by time slot to reflect Ecuadorian purchasing patterns.

In [ ]:
# Generate time slots and hours
time_slots = np.random.choice(
    ["Morning", "Afternoon", "Evening"],
    size=N,
    p=[0.30, 0.45, 0.25]
)

hours = []
for slot in time_slots:
    if slot == "Morning":
        hours.append(np.random.randint(9, 12))
    elif slot == "Afternoon":
        hours.append(np.random.randint(12, 18))
    else:
        hours.append(np.random.randint(18, 21))

minutes = np.random.randint(0, 60, size=N)
times = [f"{h:02d}:{m:02d}" for h, m in zip(hours, minutes)]

print("Time slots distribution:")
print(pd.Series(time_slots).value_counts())

In [ ]:
# Generate dates (full year 2024)
dates = pd.date_range(start="2024-01-01", end="2024-12-31", periods=N)
dates = np.random.choice(dates, size=N, replace=True)
days_of_week = [pd.Timestamp(d).day_name() for d in dates]

# Generate multiple categories per transaction
transaction_categories = []
for slot in time_slots:
    n_cats = np.random.choice([2, 3, 4], p=n_categories_prob)
    chosen = np.random.choice(
        categories,
        size=n_cats,
        replace=False,
        p=weights_by_slot[slot]
    )
    transaction_categories.append(list(chosen))

# Assemble DataFrame
transaction_ids = [f"TXN-{i:05d}" for i in range(1, N+1)]
city_col = np.random.choice(cities, size=N, p=[0.45, 0.40, 0.15])
customer_col = np.random.choice(customer_type, size=N, p=[0.70, 0.30])
payment_col = np.random.choice(payment_method, size=N, p=[0.40, 0.35, 0.25])

df = pd.DataFrame({
    "transaction_id":  transaction_ids,
    "date":            [str(d)[:10] for d in dates],
    "time":            times,
    "day_of_week":     days_of_week,
    "time_slot":       time_slots,
    "categories":      [", ".join(cats) for cats in transaction_categories],
    "city":            city_col,
    "customer_type":   customer_col,
    "payment_method":  payment_col
})

print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows:")
df.head()

## Saving to SQLite database

Data is stored in a SQLite database to simulate a real retail
data pipeline. SQL queries will be used in subsequent notebooks
to extract and transform the data.

In [ ]:
# Save to SQLite database
db_path = "../data/retail_ecuador.db"

conn = sqlite3.connect(db_path)
df.to_sql("transactions", conn, if_exists="replace", index=False)
conn.close()

print(f"Database saved at: {db_path}")
print(f"Table 'transactions' created with {N} rows")

## Summary

- 10,000 transactions successfully generated
- Each transaction contains 2 to 4 product categories
- Data reflects Ecuadorian supermarket purchasing patterns
- Stored in SQLite database: `data/retail_ecuador.db`
- Table name: `transactions`